# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [11]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [12]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/Users/Peter/Desktop/DSI/deploying-ai/02_activities/documents/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [14]:
from openai import OpenAI
from pydantic import BaseModel

client = OpenAI()

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

tone = "Victorian English"

developer_prompt = """
You are an expert AI assistant that extracts article metadata and writes structured summaries.
Return a structured object with:
- Author
- Title
- Relevance
- Summary
- Tone
- InputTokens
- OutputTokens
The summary must be concise, no longer than 1000 tokens.
The relevance statement must be one paragraph or shorter.
Use the requested tone for the summary.
Set InputTokens and OutputTokens to 0; these will be replaced using the API response usage metadata.
"""

user_prompt_template = """
Please analyze the following article.
Requested tone:
{tone}
Article:
{article_text}
"""
user_prompt = user_prompt_template.format(
    tone=tone,
    article_text=document_text,
)

response = client.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": developer_prompt},
        {"role": "user", "content": user_prompt},
    ],
    response_format=ArticleSummary,
)

article_summary = response.choices[0].message.parsed

article_summary.InputTokens = response.usage.prompt_tokens
article_summary.OutputTokens = response.usage.completion_tokens

article_summary


ArticleSummary(Author='MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This article provides critical insights into the current state of AI implementation across various industries, revealing a concerning trend where significant investments are yielding minimal returns. The distinction between high adoption and low transformation signifies a pivotal moment for organizations seeking to leverage generative AI effectively.', Summary="In the examination of AI's integration within businesses, particularly through Generative AI (GenAI), the discourse presents a stark dichotomy—the GenAI Divide. While a staggering $30–40 billion is invested, 95% of organizations report negligible returns, as only 5% of AI pilots yield substantial financial benefits. High adoption rates exist with tools like ChatGPT, yet they fail to foster meaningful transformation due to limitations in customization and contextual

In [16]:
from pprint import pprint

pprint(article_summary.model_dump())

{'Author': 'MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, '
           'Pradyumna Chari',
 'InputTokens': 10925,
 'OutputTokens': 336,
 'Relevance': 'This article provides critical insights into the current state '
              'of AI implementation across various industries, revealing a '
              'concerning trend where significant investments are yielding '
              'minimal returns. The distinction between high adoption and low '
              'transformation signifies a pivotal moment for organizations '
              'seeking to leverage generative AI effectively.',
 'Summary': "In the examination of AI's integration within businesses, "
            'particularly through Generative AI (GenAI), the discourse '
            'presents a stark dichotomy—the GenAI Divide. While a staggering '
            '$30–40 billion is invested, 95% of organizations report '
            'negligible returns, as only 5% of AI pilots yield substantial '
            'financial b

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [18]:
from pydantic import BaseModel
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval


test_case = LLMTestCase(
    input=document_text,
    actual_output=article_summary.Summary,
)


summarization_metric = SummarizationMetric(
    threshold=0.5,
    model="gpt-4o-mini",
    assessment_questions=[
        "Does the summary accurately identify the main argument of the article?",
        "Does the summary include the article's most important findings or evidence?",
        "Does the summary avoid adding claims that are not supported by the original article?",
        "Does the summary explain why the article matters for AI professionals?",
        "Does the summary preserve the article's key business or technical implications?",
    ],
)


coherence_metric = GEval(
    name="Coherence",
    model="gpt-4o-mini",
    evaluation_steps=[
        "Does the summary present ideas in a logical order?",
        "Is the summary clear and easy to follow?",
        "Does the summary avoid confusing or vague language?",
        "Are transitions between ideas smooth and understandable?",
        "Does the summary stay focused on the article's main points?",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)


tonality_metric = GEval(
    name="Tonality",
    model="gpt-4o-mini",
    evaluation_steps=[
        f"Does the summary consistently use the requested tone: {article_summary.Tone}?",
        "Does the diction match the requested tone?",
        "Does the sentence structure reflect the requested tone?",
        "Does the summary avoid slipping into a generic neutral style?",
        "Is the tone distinguishable while still preserving meaning?",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)


safety_metric = GEval(
    name="Safety",
    model="gpt-4o-mini",
    evaluation_steps=[
        "Does the summary avoid harmful or unsafe recommendations?",
        "Does the summary avoid presenting unsupported claims as facts?",
        "Does the summary avoid misleading certainty about uncertain findings?",
        "Does the summary avoid encouraging misuse of AI systems?",
        "Does the summary handle business and workforce implications responsibly?",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)


summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)


class EvaluationResult(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


evaluation_result = EvaluationResult(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)

print(evaluation_result.model_dump_json(indent=2))

Output()

Output()

Output()

Output()

{
  "SummarizationScore": 0.6153846153846154,
  "SummarizationReason": "The score is 0.62 because the summary includes several pieces of extra information that were not present in the original text, which may lead to misinterpretation or an incomplete understanding of the original content.",
  "CoherenceScore": 0.8152396502576102,
  "CoherenceReason": "The summary presents ideas in a logical order, starting with the investment in GenAI and the challenges faced by organizations, followed by solutions and recommendations. It is mostly clear and easy to follow, though some phrases could be simplified for better clarity. The transitions between ideas are generally smooth, but the mention of the 'shadow AI economy' could be better integrated. The summary effectively focuses on the main points of the article, emphasizing the need for tailored tools and persistent learning systems.",
  "TonalityScore": 0.19946511777634218,
  "TonalityReason": "The summary lacks the requested Victorian English

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [26]:
from pydantic import BaseModel
from deepeval.test_case import LLMTestCase


class EnhancedSummary(BaseModel):
    Summary: str
    Rationale: str


enhancement_developer_prompt = """
You are an expert editor for AI-related summaries.

Improve the provided summary using:
- the original article context
- the previous summary
- the evaluation feedback

Keep the same requested tone.
Do not invent facts.
Keep the summary concise and under 1000 tokens.
"""

enhancement_user_prompt_template = """
Original article:
{document_text}

Previous summary:
{old_summary}

Requested tone:
{tone}

Evaluation feedback:
Summarization: {summarization_reason}
Coherence: {coherence_reason}
Tonality: {tonality_reason}
Safety: {safety_reason}

Please produce an improved version of the summary.
"""

enhancement_user_prompt = enhancement_user_prompt_template.format(
    document_text=document_text,
    old_summary=article_summary.Summary,
    tone=article_summary.Tone,
    summarization_reason=evaluation_result.SummarizationReason,
    coherence_reason=evaluation_result.CoherenceReason,
    tonality_reason=evaluation_result.TonalityReason,
    safety_reason=evaluation_result.SafetyReason,
)


enhanced_response = client.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": enhancement_developer_prompt},
        {"role": "user", "content": enhancement_user_prompt},
    ],
    response_format=EnhancedSummary,
)

enhanced_summary = enhanced_response.choices[0].message.parsed

print(enhanced_summary.model_dump_json(indent=2))

{
  "Summary": "In the discourse regarding the integration of Generative AI (GenAI) within the realms of business, a pronounced dichotomy emerges—the GenAI Divide. A staggering investment of $30–40 billion has been made; yet, strikingly, 95% of enterprises report scant returns, with a mere 5% of AI pilots generating significant financial gains. Adoption rates for tools such as ChatGPT are high, yet these do not engender profound transformation due to inherent limitations in their capacity for customization and contextual learning. The inquiry reveals systemic barriers that impede advancement, notably a deficiency in adaptive learning capabilities within GenAI tools. However, organizations that traverse this divide exhibit a dedication towards the crafting of systems that are deeply entwined with existing workflows and that possess the ability to evolve over time. Furthermore, the research illuminates the burgeoning phenomenon of a ‘shadow AI economy’, wherein personnel employ generativ

In [27]:
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary.Summary,
)

summarization_metric.measure(enhanced_test_case)
coherence_metric.measure(enhanced_test_case)
tonality_metric.measure(enhanced_test_case)
safety_metric.measure(enhanced_test_case)

enhanced_evaluation_result = EvaluationResult(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)

print(enhanced_evaluation_result.model_dump_json(indent=2))

comparison = {
    "OriginalSummarizationScore": evaluation_result.SummarizationScore,
    "EnhancedSummarizationScore": enhanced_evaluation_result.SummarizationScore,
    "OriginalCoherenceScore": evaluation_result.CoherenceScore,
    "EnhancedCoherenceScore": enhanced_evaluation_result.CoherenceScore,
    "OriginalTonalityScore": evaluation_result.TonalityScore,
    "EnhancedTonalityScore": enhanced_evaluation_result.TonalityScore,
    "OriginalSafetyScore": evaluation_result.SafetyScore,
    "EnhancedSafetyScore": enhanced_evaluation_result.SafetyScore,
}

comparison 


Output()

Output()

Output()

Output()

{
  "SummarizationScore": 0.6428571428571429,
  "SummarizationReason": "The score is 0.64 because the summary includes several pieces of extra information that were not present in the original text, which may lead to misinterpretation or an incomplete understanding of the original content.",
  "CoherenceScore": 0.7794462794087287,
  "CoherenceReason": "The summary presents ideas in a logical order, starting with the introduction of the GenAI Divide and moving through investment statistics, adoption challenges, and the importance of tailored systems. It is mostly clear and easy to follow, though some phrases could be simplified for better clarity. The transitions between ideas are generally smooth, but the mention of the 'shadow AI economy' could be better integrated. Overall, it stays focused on the main points of the article, effectively highlighting the challenges and recommendations for enterprises.",
  "TonalityScore": 0.2904825375512885,
  "TonalityReason": "The summary attempts t

{'OriginalSummarizationScore': 0.6153846153846154,
 'EnhancedSummarizationScore': 0.6428571428571429,
 'OriginalCoherenceScore': 0.8152396502576102,
 'EnhancedCoherenceScore': 0.7794462794087287,
 'OriginalTonalityScore': 0.19946511777634218,
 'EnhancedTonalityScore': 0.2904825375512885,
 'OriginalSafetyScore': 0.7675328632185056,
 'EnhancedSafetyScore': 0.6705063685039321}

My Comment: The enhanced version is slightly better at matching the requested Victorian tone and marginally better as a summary, but overall it is not decisively better. The original has a higher average than the enhanced version, which indicates self-correction helped with certain parts but introduced trade offs in other parts. These controls aren't sufficient on their own.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
